# Тетрадь 1.0: Кредитная политика своими руками

Это первая содержательная тетрадь проекта. Мы строим мультиагентный оркестратор кредитного андеррайтинга — сервис, который принимает заявку на кредит и решает, одобрить её или нет.

Но прежде чем писать агентов, надо ответить на простой вопрос: **а по каким правилам вообще решается судьба заявки**?

Этот шаг называется **«перевод бизнес-логики в требования»**. В реальной команде его делает системный аналитик вместе с бизнесом. В нашем MVP ты делаешь его сам, потому что иначе агентам не из чего будет выбирать.

**Чему ты научишься в этой тетради:**

- что такое кредитная политика и из чего она состоит;
- какие признаки заявки банк вообще смотрит;
- что такое DTI, PTI, скоринг и почему это главные метрики в кредитовании;
- как из правил на человеческом языке получается структура данных, которую сможет использовать агент;
- как хранить правила в JSON так, чтобы их было легко менять, не переписывая код.

**Что на выходе:**

Файл `policy.json` с 30–50 правилами, который дальше будет использоваться RAG-агентом (Policy Check Agent) во второй фазе проекта.

**Пункт чеклиста, который закрывается:**

- №7 (частично): переводить бизнес-логику и требования в работающий код;
- №8: общаться с бизнес-командами и разбираться в устройстве реальных процессов;
- №11 (частично): писать парсеры данных — правила собираются из открытых источников.

**Формат работы:**

Каждый раздел — короткая теория, пример и мини-задача. В конце — большой артефакт: `policy.json`. Не пропускай мини-задачи: без них не поймёшь, что происходит.

## 1. Что такое кредитная политика

**Кредитная политика банка** — это набор правил, по которым банк решает:

1. **давать ли кредит** конкретному клиенту;
2. **на каких условиях** — ставка, срок, лимит;
3. **что делать**, если клиент не проходит по части правил.

Политика — это не «одна формула». Это десятки-сотни правил, каждое из которых описывает одну сторону риска. Условно они делятся на четыре группы:

- **Скоринговая группа.** Внутренний балл банка: сколько клиент уже пользовался кредитами, были ли просрочки, есть ли залоги.
- **Доход и платёж.** Хватает ли у клиента денег, чтобы обслуживать новый кредит. Здесь живут DTI и PTI.
- **Профиль.** Возраст, занятость, регион, гражданство, стоп-факторы.
- **Особые случаи.** Ипотека, автокредит, рефинансирование — там свои правила.

Мы строим MVP по потребительскому кредитованию, поэтому будем работать с первыми тремя группами.

**Запомни главное:** правило политики — это всегда **предикат** (утверждение, которое либо верно, либо нет) плюс **действие**, если предикат верен. Например:

```
ЕСЛИ DTI > 50%
ТО отказ с причиной «предельная долговая нагрузка»
```

Мы будем хранить правила именно в таком виде.

## 2. Из чего состоит заявка на кредит

Чтобы правило могло сработать, у заявки должны быть **признаки**, на которые правило смотрит. Ещё раз: агент не может проверить правило, если в заявке нет нужного поля.

Минимальный набор признаков для потребительского кредита:

| Признак | Тип | Что означает |
|---|---|---|
| `full_name` | string | ФИО клиента |
| `age` | int | возраст |
| `monthly_income` | float | средний месячный доход, руб |
| `loan_amount` | float | запрашиваемая сумма, руб |
| `loan_term_months` | int | срок кредита в месяцах |
| `interest_rate` | float | предполагаемая ставка, % годовых |
| `existing_monthly_debt` | float | уже существующие платежи, руб/мес |
| `credit_history_years` | float | сколько лет клиент пользуется кредитами |
| `had_delinquency` | bool | были ли просрочки |
| `employment_type` | string | `salaried` / `self_employed` / `unemployed` |
| `employment_years` | float | сколько лет работает на текущем месте |

Эти признаки мы будем генерировать в фазе 1 (тетрадь 1.3) и извлекать из PDF в extraction-агенте.

Пока запомни: **правила описываются в терминах этих признаков**.

## 3. Три ключевые метрики: DTI, PTI и ежемесячный платёж

Прежде чем писать правила, надо уметь считать три величины, на которых эти правила стоят.

**Ежемесячный платёж по кредиту** (аннуитетный):

$$
M = P \cdot \frac{i \cdot (1+i)^n}{(1+i)^n - 1}
$$

где:
- `P` — сумма кредита;
- `i` — месячная ставка (`годовая / 12 / 100`);
- `n` — срок в месяцах.

**DTI (Debt-to-Income)** — долговая нагрузка:

$$
\text{DTI} = \frac{\text{ежемесячный платёж по новому кредиту} + \text{существующие платежи}}{\text{месячный доход}}
$$

**PTI (Payment-to-Income)** — нагрузка только по новому кредиту:

$$
\text{PTI} = \frac{\text{ежемесячный платёж по новому кредиту}}{\text{месячный доход}}
$$

Типичные пороги для российского потребительского кредитования (проверишь по открытым источникам в задаче 1.0.4):

- **DTI > 50%** — отказ;
- **DTI 40–50%** — снижение суммы или ручная проверка;
- **PTI > 40%** — почти всегда отказ.

In [10]:
import json
from pathlib import Path

# Создаём папку data/, если её ещё нет.
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

print("Папка data/ готова:", DATA_DIR.resolve())

Папка data/ готова: /Users/alexander/Developer/underwriting-orchestrator/data


### Мини-задача 3.1: напиши калькулятор аннуитетного платежа

Реализуй функцию `monthly_payment(loan_amount, annual_rate, term_months)`, которая возвращает ежемесячный платёж по формуле выше.

Проверь на примере:

- сумма: 500 000 ₽
- ставка: 18% годовых
- срок: 36 месяцев

Ожидаемый ответ — около **18 078 ₽** (может отличаться на пару рублей — это нормально, зависит от округления).

Подсказка: сначала переведи годовую ставку в месячную, потом подставь всё в формулу.

In [12]:
def monthly_payment(loan_amount, annual_rate, term_months):
    mon_rate = annual_rate/12/100
    return loan_amount * mon_rate*(mon_rate+1)**term_months/((mon_rate+1)**term_months-1)

print(monthly_payment(500000,18,36))

18076.197767958507


### Мини-задача 3.2: напиши калькуляторы DTI и PTI

1. `pti(monthly_payment, monthly_income)` — возвращает долю (0..1+), которую платёж составляет от дохода.
2. `dti(monthly_payment, existing_debt, monthly_income)` — то же, но с учётом существующих платежей.

Проверь:

- доход 80 000 ₽, платёж 18 078 ₽, существующих долгов 5 000 ₽ →
  - PTI ≈ 0.226 (22.6%)
  - DTI ≈ 0.288 (28.8%)

In [13]:
def pti(monthly_payment, monthly_income):
    return monthly_payment / monthly_income

def dti(monthly_payment, existing_debt, monthly_income):
    return (monthly_payment+existing_debt)/monthly_income

print(pti(18078,80000))
print(dti(18078,5000,80000))

0.225975
0.288475


## 4. Формат хранения правила

Мы будем хранить каждое правило как **JSON-объект** со следующими полями:

```json
{
  "id": "DTI_50",
  "category": "income",
  "description": "Если DTI больше 50%, выдать отказ.",
  "condition": {
    "metric": "dti",
    "operator": ">",
    "value": 0.5
  },
  "action": "reject",
  "reason": "Предельная долговая нагрузка (DTI > 50%)",
  "severity": "high"
}
```

Разберём поля:

- `id` — уникальное имя правила, по нему на него ссылаются;
- `category` — группа (`income`, `profile`, `credit_history`, `special`);
- `description` — правило на человеческом языке. Это пойдёт в промпт LLM;
- `condition` — машиночитаемая часть:
  - `metric` — на какую величину смотрим (это либо признак заявки, либо вычисляемая метрика типа `dti`);
  - `operator` — `>`, `>=`, `<`, `<=`, `==`, `in`, `not_in`;
  - `value` — с чем сравниваем;
- `action` — что делать: `approve`, `reject`, `review`, `lower_amount`, `info`;
- `reason` — почему так решили, эту причину увидит аналитик в финальном отчёте;
- `severity` — `high` / `medium` / `low`, влияет на приоритет при конфликте правил.

**Почему именно так?** Потому что агенту нужны **два представления**:

1. **Текст** (`description`) — чтобы LLM могла прочитать и интерпретировать в сложных случаях.
2. **Формальное условие** (`condition`) — чтобы код мог проверить быстро и без ошибок.

Это классический приём в AI-системах: правила, записанные в двух формах, — «текст для человека, код для машины».

### Мини-задача 4.1: напиши функцию, которая проверяет одно правило

Функция `check_rule(rule, application_metrics)` принимает:

- `rule` — словарь с правилом из JSON;
- `application_metrics` — словарь с вычисленными метриками заявки (там есть ключи `dti`, `pti`, `age`, `monthly_income` и т. п.).

Возвращает `True`, если условие выполнено, `False`, если нет.

Поддержи операторы `>`, `>=`, `<`, `<=`, `==`.

Проверь:

```python
rule = {"condition": {"metric": "dti", "operator": ">", "value": 0.5}}
metrics = {"dti": 0.62}
check_rule(rule, metrics)  # должно быть True

metrics2 = {"dti": 0.31}
check_rule(rule, metrics2)  # должно быть False
```

Подумай, что должно случиться, если в `metrics` нет нужного ключа. Правильно — выбросить понятную ошибку. Не «молча False».

In [30]:
rule = {"condition": {"metric": "dti", "operator": "<", "value": 0.5}}
metrics1 = {"dti": 0.62}
metrics2 = {"dti": 0.31}
#check_rule(rule, metrics1)  # должно быть True
#check_rule(rule, metrics2)
def check_rule(rule,metrics):
    if rule['condition']['metric'] not in metrics.keys():
        raise ValueError("Metrics cannot be applied to the rule.")
    if rule['condition']['operator'] == '>':
        return False if rule['condition']['value'] > metrics['dti'] else True
    elif rule['condition']['operator'] == '<':
        return False if rule['condition']['value'] < metrics['dti'] else True
    elif rule['condition']['operator'] == '==':
        return True if rule['condition']['value'] == metrics['dti'] else False # остальное некогда писать принцип я понял
    #return True if rule['condition']['metric'] else False
print(check_rule(rule, metrics1))
print(check_rule(rule, metrics2))

False
True


## 5. Исследование: собери политику из открытых источников

Теперь самое интересное. **Реальные банки не публикуют свою политику целиком.** Но по открытым источникам (сайты банков, 
статьи, аналитика ЦБ РФ) можно восстановить **типичные правила**, которые используют российские банки для потребительского кредитования.

Твоя задача — собрать из открытых источников **30–50 правил** и записать их в формате из раздела 4.

**Где искать:**

- сайт ЦБ РФ — там публикуются нормативы и рекомендации;
- официальные страницы Альфа-Банка, Сбера, ВТБ про потребительские кредиты — там указаны требования к клиенту;
- статьи аналитиков (`banki.ru`, `frankrg.com`, `raexpert.ru`) — там разбираются типичные скоринговые правила.

**Что искать в первую очередь:**

- требования к возрасту (минимум/максимум);
- требования к доходу (минимум в месяц);
- требования к занятости (стаж на последнем месте);
- пороги DTI и PTI;
- правила по кредитной истории (просрочки, количество активных кредитов);
- ограничения по сумме в зависимости от дохода;
- базовые стоп-факторы (нет гражданства РФ, младше 21 года, старше 70, безработный и т. д.).

**Важно:** не выдумывай правила «от себя». Каждое правило должно иметь источник. Даже если это статья на `banki.ru` — впиши ссылку в описание. Это называется **traceability** (прослеживаемость), и в реальной работе это требование №1.

### Мини-задача 5.1: собери 30 правил в файл policy.json

Сформируй файл `data/policy.json` — список правил. Каждое правило — объект по схеме из раздела 4.

Минимум — по 5 правил на каждую из четырёх категорий (`income`, `profile`, `credit_history`, `special`), итого минимум 20. Хорошая работа — 30–40 правил с реальными порогами и осмысленными причинами.

**Проверка:** после того, как ты соберёшь файл, открой его программно и убедись, что каждое правило содержит все обязательные поля.

Подсказка: пиши JSON не руками, а генератором ниже. Так его легче редактировать, и не будет синтаксических ошибок.

In [ ]:
# ТВОЙ КОД ЗДЕСЬ

# Пример шаблона одного правила:
example_rule = {
    "id": "AGE_MIN_21",
    "category": "profile",
    "description": "Минимальный возраст заёмщика — 21 год.",
    "condition": {"metric": "age", "operator": ">=", "value": 21},
    "action": "reject",
    "reason": "Возраст меньше минимально допустимого",
    "severity": "high",
    "source": "https://www.sberbank.ru/ru/person/credits/money",
}

# Собери свой список из 30 таких объектов и запиши в data/policy.json.


### Мини-задача 5.2: валидация

Напиши функцию `validate_policy(policy)`, которая:

1. Проверяет, что каждое правило — словарь с обязательными полями: `id`, `category`, `description`, `condition`, `action`, `reason`, `severity`.
2. Проверяет, что `condition` содержит `metric`, `operator`, `value`.
3. Проверяет, что `operator` — один из допустимых.
4. Проверяет, что `action` — один из допустимых.
5. Проверяет, что все `id` уникальны.
6. Возвращает список ошибок. Если он пуст — политика валидна.

Прогони её на своём `policy.json`. Если есть ошибки — исправь.

In [40]:
import json

# Open and load JSON file into a dictionary
with open('data/policy.json', 'r') as file:
    policy = json.load(file)

# Now 'data' is a dictionary
#print(type(data), len(data), '\n', data[0])  # <class 'dict'>

In [46]:
def validate_policy(policy, verbose=True):
    """
    Проверяет политику на корректность. Возвращает (ok, errors).
    ok     : True, если ошибок нет
    errors : список сообщений об ошибках
    """
    required_rule_fields = {"id", "category", "description",
                            "condition", "action", "reason",
                            "severity", "source"}
    required_condition_fields = {"metric", "operator", "value"}

    valid_operators = {">", ">=", "<", "<=", "==", "in", "not_in"}
    valid_actions = {"approve", "reject", "review", "lower_amount", "info"}
    valid_severities = {"high", "medium", "low"}
    valid_categories = {"income", "profile", "credit_history", "special"}

    errors = []
    seen_ids = set()

    if not isinstance(policy, list):
        return False, ["policy должен быть списком правил"]

    for i, rule in enumerate(policy):
        prefix = f"Правило #{i}"

        # 0. Вообще словарь ли это
        if not isinstance(rule, dict):
            errors.append(f"{prefix}: не словарь, а {type(rule).__name__}")
            continue

        rid = rule.get("id", "<без id>")
        prefix = f"Правило '{rid}'"

        # 1. Обязательные поля правила
        missing = required_rule_fields - set(rule.keys())
        if missing:
            errors.append(f"{prefix}: не хватает полей {sorted(missing)}")
            continue  # дальше проверять нечего

        # 2. Уникальность id
        if rid in seen_ids:
            errors.append(f"{prefix}: дубликат id")
        seen_ids.add(rid)

        # 3. category
        if rule["category"] not in valid_categories:
            errors.append(
                f"{prefix}: неизвестная category '{rule['category']}'. "
                f"Допустимо: {sorted(valid_categories)}"
            )

        # 4. action
        if rule["action"] not in valid_actions:
            errors.append(
                f"{prefix}: неизвестная action '{rule['action']}'. "
                f"Допустимо: {sorted(valid_actions)}"
            )

        # 5. severity
        if rule["severity"] not in valid_severities:
            errors.append(
                f"{prefix}: неизвестная severity '{rule['severity']}'. "
                f"Допустимо: {sorted(valid_severities)}"
            )

        # 6. description и reason — непустые строки
        for f in ("description", "reason"):
            if not isinstance(rule[f], str) or not rule[f].strip():
                errors.append(f"{prefix}: поле '{f}' пустое или не строка")

        # 7. condition
        cond = rule["condition"]
        if not isinstance(cond, dict):
            errors.append(f"{prefix}: condition должен быть словарём")
            continue

        missing_cond = required_condition_fields - set(cond.keys())
        if missing_cond:
            errors.append(
                f"{prefix}: в condition не хватает полей {sorted(missing_cond)}"
            )
            continue

        if cond["operator"] not in valid_operators:
            errors.append(
                f"{prefix}: неизвестный operator '{cond['operator']}'. "
                f"Допустимо: {sorted(valid_operators)}"
            )

        if not isinstance(cond["metric"], str) or not cond["metric"].strip():
            errors.append(f"{prefix}: metric должен быть непустой строкой")

        # value: для in/not_in — список, для остальных — число или строка
        val = cond["value"]
        if cond["operator"] in ("in", "not_in"):
            if not isinstance(val, (list, tuple, set)):
                errors.append(
                    f"{prefix}: для operator '{cond['operator']}' "
                    f"value должен быть списком, а не {type(val).__name__}"
                )
        else:
            if not isinstance(val, (int, float, str, bool)):
                errors.append(
                    f"{prefix}: value должен быть числом или строкой, "
                    f"а не {type(val).__name__}"
                )

    ok = len(errors) == 0
    if verbose:
        if ok:
            print(f"✅ policy валидна: {len(policy)} правил, ошибок нет")
        else:
            print(f"❌ найдено ошибок: {len(errors)}")
            for e in errors:
                print("   -", e)
    return ok, errors

print(validate_policy(policy))

✅ policy валидна: 50 правил, ошибок нет
(True, [])


## 6. Как эту политику будет использовать агент

Здесь мы немного забегаем вперёд, чтобы ты видел цель.

В фазе 2 у нас будет **Policy Check Agent**. Он работает так:

1. Берёт заявку и вычисляет её метрики (`dti`, `pti`, `age`, `monthly_income` и т. п.).
2. Для каждой метрики достаёт из `policy.json` все правила, которые на неё смотрят. Это делает **RAG-поиск** по описаниям правил и по метрикам.
3. Передаёт найденные правила и метрики в LLM.
4. LLM проверяет: какие правила сработали, какие нет, какие конфликтуют. Возвращает структурированный ответ.
5. Результат пишется в БД и идёт дальше — в Pricing Agent и Brief Agent.

**Почему RAG, а не просто «загрузить всё в промпт»?** Потому что правил может быть много, а контекст LLM ограничен. Мы отбираем только релевантные — по метрике (`dti`, `age` и т. д.). Это классический подход «retrieve → augment → generate».

**Твоя работа в этой тетради — шаг 1 и часть шага 2.** Всё, что идёт дальше, — в фазе 2.

Запомни: **если `policy.json` плохой, то и агент будет работать плохо**. Нет «умного LLM», который спасёт кривую политику. Поэтому сейчас ты делаешь самую важную часть проекта.

## 7. Артефакт этой тетради

В конце у тебя должно быть:

1. **`data/policy.json`** — файл с 30+ правилами.
2. **Функция `validate_policy`** — она либо внутри этой тетради, либо в отдельном скрипте `scripts/validate_policy.py`.
3. **Функция `check_rule`** — пригодится дальше в агенте.
4. **Список источников** — файл `data/policy_sources.md` со ссылками, откуда взято каждое правило. Это требование traceability.

**Что делаем дальше:** в тетради 1.1 ты возьмёшь эти правила и дополнишь их ещё несколькими, уже спарсив сайты автоматически через `requests` + `BeautifulSoup`. Это будет твоя первая практика с парсерами.

## Что закрыто по чеклисту вакансии

После прохождения этой тетради у тебя отработаны (полностью или частично):

- **№7**: перевод бизнес-логики в код — правила на человеческом языке превращены в JSON и в функции-предикаты.
- **№8**: разбираешься в реальных процессах — читал кредитную политику банков, разбирался в DTI/PTI, скоринге.
- **№11** (начало): данные собираются из открытых источников — пока руками, в 1.1 будем автоматизировать.

**Что я ожидаю от тебя после этой тетради:**

1. Файл `data/policy.json` — твой, а не скопированный.
2. Файл `data/policy_sources.md` — со ссылками.
3. Короткий ответ на вопрос: «Почему кредитная политика не может быть одной формулой, а обязана состоять из десятков правил?» Этот ответ потом пойдёт в документацию проекта.

Когда закончишь — покажи мне артефакты, и я дам следующую тетрадь (1.1 — парсинг открытых источников).